In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import matplotlib.pyplot as plt

## RATING MAP
- A dictionary used to convert word ratings like "three" to an integer. Thus making the rating numerical for analysis

In [4]:
Rating_map ={
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5
}

In [29]:
# Data collection from the 50 pages
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
MAX_PAGES = 50

In [32]:
def scrape_books():
    """Iterates through pages and extracts book data."""
    all_books_data = []
    print(f"Starting to scrape {MAX_PAGES} pages...")

    for page_num in range(1, MAX_PAGES + 1):
        url = BASE_URL.format(page_num)
        print(f"Scraping page {page_num}...")

        try:
            response = requests.get(url)
            response.raise_for_status() # Check for bad status codes
            soup = BeautifulSoup(response.text, 'html.parser')
            articles = soup.find_all('article', class_='product_pod')

            for article in articles:
                # 1. Title
                title = article.h3.a['title']

                # 2. Price (includes currency symbol)
                price_text = article.find('p', class_='price_color').text

                # 3. Availability Status
                availability = article.find('p', class_='instock availability').text.strip()

                # 4. Rating (in text format like 'star-rating Three')
                rating_class = article.find('p', class_=re.compile(r'star-rating'))['class'][1]
                
                all_books_data.append({
                    'Title': title,
                    'Price_Raw': price_text,
                    'Availability': availability,
                    'Rating_Text': rating_class
                })

            time.sleep(0.5) # Be respectful: small delay between requests
        
        except requests.exceptions.RequestException as e:
            print(f"Error scraping page {page_num}: {e}")
            break